In Part 1 we saw the raw speed gap: **Python OOP ≈ 3.4 s**, **NumPy ≈ 0.07 s**, **C++ ≈ 0.005 s** for 10k particles × 1k steps.

But production quant desks and game studios don't just rewrite everything in C++. They need:

1. **Incremental migration** — keep the Python research API, swap the hot loop to C++
2. **SIMD vectorization** — process 8 floats at once with AVX2
3. **Cache-line alignment** — guarantee that data lands in L1 efficiently
4. **A real pricing kernel** — Monte Carlo option pricing, the "hello world" of quant finance

This part teaches the **industry bridge**: `pybind11` + SIMD + alignment + a real derivative-pricing example.

## 1. pybind11 — The Glue Between Research and Production

`pybind11` lets you write C++ functions/classes and import them in Python as if they were native modules.

**Why it's the industry standard:**
- Zero-copy NumPy interoperability (`py::array_t`)
- Exposes C++ `class`es with Pythonic syntax
- Header-only: just `pip install pybind11`

**The mental model:**


Research script ----calls----> particle_cpp.so (C++ compiled)  | |  Python API                    pybind11 wrapper  |  raw C++ loop




We will expose a `ParticleSystem` class with:
- `__init__(n)` — allocate
- `step(dt)` — update
- `get_positions()` — return NumPy array (zero-copy view)

Save this as  `particle_pybind.cpp`

In [ ]:
#include <pybind11/pybind11.h>
#include <pybind11/numpy.h>
#include <vector>
#include <random>

namespace py = pybind11;

struct Particle {
    float x, y, vx, vy;
};

class ParticleSystem {
    std::vector<Particle> particles;
public:
    ParticleSystem(int n) : particles(n) {
        std::mt19937 rng(42);
        std::uniform_real_distribution<float> dist(0.0f, 1.0f);
        for (auto& p : particles) {
            p.x = dist(rng);
            p.y = dist(rng);
            p.vx = dist(rng) - 0.5f;
            p.vy = dist(rng) - 0.5f;
        }
    }

    void step(float dt) {
        for (auto& p : particles) {
            p.x += p.vx * dt;
            p.y += p.vy * dt;
            if (p.x < 0.0f || p.x > 1.0f) p.vx *= -1.0f;
            if (p.y < 0.0f || p.y > 1.0f) p.vy *= -1.0f;
        }
    }

    py::array_t<float> get_positions() {
        // Return a NumPy array of shape (N, 2) — zero copy from C++ memory
        py::array_t<float> result({(int)particles.size(), 2});
        auto buf = result.mutable_unchecked<2>();
        for (size_t i = 0; i < particles.size(); ++i) {
            buf(i, 0) = particles[i].x;
            buf(i, 1) = particles[i].y;
        }
        return result;
    }
};

PYBIND11_MODULE(particle_cpp, m) {
    py::class_<ParticleSystem>(m, "ParticleSystem")
        .def(py::init<int>())
        .def("step", &ParticleSystem::step)
        .def("get_positions", &ParticleSystem::get_positions);
}

### Compilation (run in terminal)

Make sure `pybind11` is installed:

```bash
pip install pybind11
```

Compile the shared object (Linux/macOS):

```bash
c++ -O3 -Wall -shared -std=c++11 -fPIC \
    $(python3 -m pybind11 --includes) \
    particle_pybind.cpp \
    -o particle_cpp$(python3-config --extension-suffix)
```
On Windows (MSVC)

```cmd
cl /O2 /W4 /MD /EHsc particle_pybind.cpp /I%PYBIND11_INCLUDE% /I%PYTHON_INCLUDE% /link /DLL /OUT:particle_cpp.pyd
```

The output file ( `particle_cpp*.so  or  particle_cpp.pyd` ) is imported like any Python module.

In [ ]:
# Make sure particle_cpp*.so is in this directory or in PYTHONPATH
import time
import particle_cpp  # <-- our C++ module

n = 10_000
steps = 1_000
dt = 0.01

sys = particle_cpp.ParticleSystem(n)

t0 = time.perf_counter()
for _ in range(steps):
    sys.step(dt)
t1 = time.perf_counter()

pos = sys.get_positions()
print(f"pybind11 C++: {t1-t0:.4f}s | First particle: ({pos[0,0]:.4f}, {pos[0,1]:.4f})")

## 2. SIMD with AVX2 — Eight Particles in One Breath

CPUs have **vector registers** (256-bit wide on modern x86). One AVX2 register holds **8 floats** (32 bits each).

Instead of:
```cpp
for each particle:
    p.x += p.vx * dt;   // 1 FLOP, scalar


We do:

```cpp
__m256 xv = _mm256_loadu_ps(x + i);      // load 8 x's
__m256 vxv = _mm256_loadu_ps(vx + i);    // load 8 vx's
xv = _mm256_add_ps(xv, _mm256_mul_ps(vxv, dt_vec));  // 8 FLOPs at once
```

*Theoretical throughput:*
 
AVX2  `vaddps`  +  `vmulps`  throughput ≈ 2–3 cycles per instruction on modern cores
 
1. Each instruction processes 8 floats
2. Scalar loop: ~4–6 cycles per particle
3. AVX2 loop: ~4–6 cycles per 8 particles → ~8× speedup

*But there's a catch*: SIMD likes *Structure-of-Arrays (SoA)*.
 
AoS:  `Particle {x,y,vx,vy}`  — mixed data, hard to load 8 x's contiguously
 
SoA: separate arrays  `x[], y[], vx[], vy[]`  — perfect for  `_mm256_loadu_ps`

*Formula for vectorized throughput:*

$FLOPs/cycle= \frac{vector\ width}{scalar\ width}\ \times \ issue\ rate\ =\ \frac{256}{32}\ \times\ 2\ =\ 16\ FLOPs/cycle$

Save as `particle_avx2.cpp`

In [ ]:
#include <immintrin.h>
#include <vector>
#include <random>

// SoA layout: separate contiguous arrays
struct ParticleSoA {
    std::vector<float> x, y, vx, vy;
    ParticleSoA(int n) : x(n), y(n), vx(n), vy(n) {
        std::mt19937 rng(42);
        std::uniform_real_distribution<float> dist(0.0f, 1.0f);
        for (int i = 0; i < n; ++i) {
            x[i] = dist(rng); y[i] = dist(rng);
            vx[i] = dist(rng) - 0.5f; vy[i] = dist(rng) - 0.5f;
        }
    }
};

void step_avx2(ParticleSoA& p, int n, float dt) {
    __m256 dt_vec = _mm256_set1_ps(dt);
    __m256 zero = _mm256_set1_ps(0.0f);
    __m256 one = _mm256_set1_ps(1.0f);
    __m256 neg = _mm256_set1_ps(-1.0f);

    for (int i = 0; i < n; i += 8) {
        // Load 8-wide chunks
        __m256 xv = _mm256_loadu_ps(&p.x[i]);
        __m256 yv = _mm256_loadu_ps(&p.y[i]);
        __m256 vxv = _mm256_loadu_ps(&p.vx[i]);
        __m256 vyv = _mm256_loadu_ps(&p.vy[i]);

        // Update position: x += vx * dt
        xv = _mm256_add_ps(xv, _mm256_mul_ps(vxv, dt_vec));
        yv = _mm256_add_ps(yv, _mm256_mul_ps(vyv, dt_vec));

        // Bounce masks
        __m256 mx = _mm256_or_ps(
            _mm256_cmp_ps(xv, zero, _CMP_LT_OS),
            _mm256_cmp_ps(xv, one,  _CMP_GT_OS)
        );
        __m256 my = _mm256_or_ps(
            _mm256_cmp_ps(yv, zero, _CMP_LT_OS),
            _mm256_cmp_ps(yv, one,  _CMP_GT_OS)
        );

        // If mask true, flip velocity
        vxv = _mm256_blendv_ps(vxv, _mm256_mul_ps(vxv, neg), mx);
        vyv = _mm256_blendv_ps(vyv, _mm256_mul_ps(vyv, neg), my);

        // Store back
        _mm256_storeu_ps(&p.x[i], xv);
        _mm256_storeu_ps(&p.y[i], yv);
        _mm256_storeu_ps(&p.vx[i], vxv);
        _mm256_storeu_ps(&p.vy[i], vyv);
    }
}

## 3. Memory Alignment — Why `alignas(64)` Matters

A **cache line** is 64 bytes on virtually every modern CPU. If your array starts at an address divisible by 64, an entire SIMD load (`_mm256_load_ps`) stays within **one** cache line.

**Misalignment penalty:**
- `_mm256_loadu_ps` (unaligned) → hardware handles crossing cache lines, but adds 1–5 cycles
- `_mm256_load_ps` (aligned) → guaranteed single cache-line hit

**The formula:**
$$\text{effective latency} = \begin{cases} 
L1 & \text{if addr} \equiv 0 \pmod{64} \\
L1 + \text{cross-line penalty} & \text{otherwise}
\end{cases}$$


Save as  `particle_aligned.cpp`

In [ ]:
#include <immintrin.h>
#include <cstdlib>

// Aligned allocator for std::vector
template <typename T, std::size_t Alignment = 64>
class AlignedAllocator {
public:
    using value_type = T;
    T* allocate(std::size_t n) {
        void* ptr = nullptr;
        // C++17: ptr = std::aligned_alloc(Alignment, n * sizeof(T));
        // Portable AVX way:
        ptr = _mm_malloc(n * sizeof(T), Alignment);
        return static_cast<T*>(ptr);
    }
    void deallocate(T* ptr, std::size_t) { _mm_free(ptr); }
};

// Usage: std::vector<float, AlignedAllocator<float>> x(n);
// This guarantees x.data() % 64 == 0

// Aligned AVX2 step (uses _mm256_load_ps instead of _mm256_loadu_ps)
void step_avx2_aligned(float* __restrict x, float* __restrict y,
                       float* __restrict vx, float* __restrict vy,
                       int n, float dt) {
    __m256 dt_vec = _mm256_set1_ps(dt);
    __m256 zero = _mm256_set1_ps(0.0f);
    __m256 one = _mm256_set1_ps(1.0f);
    __m256 neg = _mm256_set1_ps(-1.0f);

    for (int i = 0; i < n; i += 8) {
        __m256 xv = _mm256_load_ps(x + i);   // aligned!
        __m256 yv = _mm256_load_ps(y + i);
        __m256 vxv = _mm256_load_ps(vx + i);
        __m256 vyv = _mm256_load_ps(vy + i);

        xv = _mm256_add_ps(xv, _mm256_mul_ps(vxv, dt_vec));
        yv = _mm256_add_ps(yv, _mm256_mul_ps(vyv, dt_vec));

        __m256 mx = _mm256_or_ps(
            _mm256_cmp_ps(xv, zero, _CMP_LT_OS),
            _mm256_cmp_ps(xv, one,  _CMP_GT_OS)
        );
        __m256 my = _mm256_or_ps(
            _mm256_cmp_ps(yv, zero, _CMP_LT_OS),
            _mm256_cmp_ps(yv, one,  _CMP_GT_OS)
        );

        vxv = _mm256_blendv_ps(vxv, _mm256_mul_ps(vxv, neg), mx);
        vyv = _mm256_blendv_ps(vyv, _mm256_mul_ps(vyv, neg), my);

        _mm256_store_ps(x + i, xv);   // aligned store
        _mm256_store_ps(y + i, yv);
        _mm256_store_ps(vx + i, vxv);
        _mm256_store_ps(vy + i, vyv);
    }
}

## 4. Real Quant Example — Monte Carlo European Call Pricing

This is the "industry hello world" of computational finance.

**Black-Scholes Monte Carlo:**

Under risk-neutral measure, the stock price at maturity $T$ is:

$$S_T = S_0 \exp\left(\left(r - \frac{\sigma^2}{2}\right)T + \sigma\sqrt{T}Z\right)$$

where $Z \sim \mathcal{N}(0,1)$.

The Monte Carlo estimate of a European call option price is:

$$\hat{C} = e^{-rT} \cdot \frac{1}{N} \sum_{i=1}^{N} \max(S_T^{(i)} - K, 0)$$

**Why this matters in production:**
- A desk might run 10 million paths overnight for exotic options
- Python `for` loops here are unusable (minutes vs seconds)
- C++ with `pybind11` lets quants keep their Python backtest framework while calling a compiled pricer

We will build:
1. Python baseline (NumPy)
2. C++ kernel exposed via `pybind11`

In [ ]:
import numpy as np
import time

def mc_call_numpy(S0=100.0, K=100.0, T=1.0, r=0.05, sigma=0.2, n_paths=1_000_000):
    """Vectorized Monte Carlo European Call in pure Python/NumPy."""
    rng = np.random.default_rng(42)
    Z = rng.standard_normal(n_paths)
    drift = (r - 0.5 * sigma**2) * T
    diffusion = sigma * np.sqrt(T)
    ST = S0 * np.exp(drift + diffusion * Z)
    payoff = np.maximum(ST - K, 0.0)
    price = np.exp(-r * T) * np.mean(payoff)
    return price

# Warmup + timing
_ = mc_call_numpy(n_paths=100_000)

t0 = time.perf_counter()
price_py = mc_call_numpy(n_paths=2_000_000)
t1 = time.perf_counter()

print(f"NumPy MC price: {price_py:.4f}")
print(f"NumPy MC time:  {t1-t0:.4f}s")

Save as  `quant_pybind.cpp`

In [ ]:
#include <pybind11/pybind11.h>
#include <random>
#include <cmath>

double mc_european_call(double S0, double K, double T,
                        double r, double sigma, int n_paths) {
    std::mt19937 rng(42);
    std::normal_distribution<double> dist(0.0, 1.0);

    const double drift = (r - 0.5 * sigma * sigma) * T;
    const double diffusion = sigma * std::sqrt(T);
    double sum = 0.0;

    for (int i = 0; i < n_paths; ++i) {
        double Z = dist(rng);
        double ST = S0 * std::exp(drift + diffusion * Z);
        sum += std::max(ST - K, 0.0);
    }

    return std::exp(-r * T) * (sum / n_paths);
}

namespace py = pybind11;
PYBIND11_MODULE(quant_cpp, m) {
    m.doc() = "Minimal quant kernels";
    m.def("mc_european_call", &mc_european_call,
          "Monte Carlo European Call (C++ scalar)",
          py::arg("S0"), py::arg("K"), py::arg("T"),
          py::arg("r"), py::arg("sigma"), py::arg("n_paths"));
}

Compile:

```bash
c++ -O3 -shared -std=c++11 -fPIC \
    $(python3 -m pybind11 --includes) \
    quant_pybind.cpp -o quant_cpp$(python3-config --extension-suffix)
```

Python usage:

```python
import quant_cpp
price_cpp = quant_cpp.mc_european_call(100.0, 100.0, 1.0, 0.05, 0.2, 2_000_000)
```

## 5. Performance Summary & Rules of Thumb

| Layer | Technology | When to Use | Typical Speedup |
|-------|-----------|-------------|-----------------|
| Research | Python OOP | Idea validation, readability | 1× |
| Prototype | NumPy / Pandas | Matrix ops, vectorized math | 10–100× |
| Hybrid | `pybind11` + C++ | Custom loops, state machines | 50–500× |
| Kernel | C++ AVX2 | Hot loops, particle systems | 200–2000× |
| Extreme | C++ AVX-512 / CUDA | Massively parallel, GPUs | 1000–10000× |

**The decision tree used on quant desks:**

Is it a matrix multiply?  YES → NumPy / CuBLAS  NO → Is it a tight loop with state?  YES → Can it vectorize?  YES → C++ AVX2  NO → C++ scalar + pybind11  NO → Keep in Python

**Key formulas to remember:**

1. **Amdahl's Law** (where to optimize):
   $$T_{\text{new}} = T_{\text{old}} \cdot \left[(1 - P) + \frac{P}{S}\right]$$
   If only 20% of your code is in C++ ($P=0.2$) and C++ is 100× faster ($S=100$):
   $$T_{\text{new}} = 0.8 + \frac{0.2}{100} = 0.802 \Rightarrow \text{only 1.25× total speedup}$$
   **Lesson:** Profile first. Move the *actual* bottleneck to C++.

2. **SIMD throughput ceiling:**
   $$\text{Max GFLOPs} = \text{cores} \times \text{freq} \times \frac{\text{vector width}}{32} \times 2 \text{ (FMA)}$$
   For 8-core @ 3.5 GHz, AVX2: $8 \times 3.5 \times 8 \times 2 = 448$ GFLOPs theoretical.

3. **Memory wall:**
   If your algorithm is **memory-bound** (not compute-bound), SIMD won't help. Bandwidth ≈ 50 GB/s. If you touch 64 bytes per particle, max particles/sec ≈ 780 million. Math-bound kernels (MC, matrix mult) benefit most from SIMD.


Run this after compiling both C++ modules to see the full comparison.

In [ ]:
import time
import numpy as np

# ---------------------------------------------------------
# 1. PYTHON NUMPY BASELINE (Particle system)
# ---------------------------------------------------------
def run_numpy(n=10_000, steps=1_000, dt=0.01):
    rng = np.random.default_rng(42)
    x = rng.random(n)
    y = rng.random(n)
    vx = rng.random(n) - 0.5
    vy = rng.random(n) - 0.5
    t0 = time.perf_counter()
    for _ in range(steps):
        x += vx * dt
        y += vy * dt
        vx = np.where((x < 0) | (x > 1), -vx, vx)
        vy = np.where((y < 0) | (y > 1), -vy, vy)
    return time.perf_counter() - t0

# ---------------------------------------------------------
# 2. PYTHON NUMPY BASELINE (Monte Carlo)
# ---------------------------------------------------------
def run_mc_numpy(n_paths=2_000_000):
    rng = np.random.default_rng(42)
    Z = rng.standard_normal(n_paths)
    drift = (0.05 - 0.5 * 0.2**2) * 1.0
    diffusion = 0.2 * np.sqrt(1.0)
    ST = 100.0 * np.exp(drift + diffusion * Z)
    payoff = np.maximum(ST - 100.0, 0.0)
    return np.exp(-0.05) * np.mean(payoff)

# ---------------------------------------------------------
# 3. C++ MODULES (uncomment after compiling)
# ---------------------------------------------------------
# import particle_cpp
# import quant_cpp

# ---------------------------------------------------------
# TIMING
# ---------------------------------------------------------
print("=" * 60)
print("PARTICLE SYSTEM BENCHMARK")
print("=" * 60)
t_py = run_numpy()
print(f"Python NumPy:  {t_py:.4f}s")

# After compiling particle_cpp:
# sys = particle_cpp.ParticleSystem(10_000)
# t0 = time.perf_counter()
# for _ in range(1000): sys.step(0.01)
# t_cpp = time.perf_counter() - t0
# print(f"pybind11 C++:  {t_cpp:.4f}s  ({t_py/t_cpp:.1f}x)")

print("\n" + "=" * 60)
print("MONTE CARLO BENCHMARK")
print("=" * 60)
t0 = time.perf_counter()
p = run_mc_numpy()
t1 = time.perf_counter()
print(f"NumPy MC:      {t1-t0:.4f}s  (price={p:.4f})")

# After compiling quant_cpp:
# t0 = time.perf_counter()
# p2 = quant_cpp.mc_european_call(100,100,1,0.05,0.2,2_000_000)
# t1 = time.perf_counter()
# print(f"pybind11 MC:   {t1-t0:.4f}s  (price={p2:.4f})  ({(t1-t0)/(t1-t0):.1f}x)")


Homework for Part 3
1. Compile and run both  particle_cpp  and  quant_cpp . Fill in the benchmark script above with real timings.
2. Profile the C++ compile flags: try  -O2  vs  -O3  vs  -O3 -march=native . How much does  -march=native  help on your CPU?
3. AVX2 exercise: Modify  particle_pybind.cpp  to use the AVX2 SoA kernel from Cell 7. Expose it as  ParticleSystemAVX2  in Python. Time it against the scalar C++ version.
4. Read about:  std::execution::par  (C++17 parallel algorithms). Replace the scalar  for  loop in the Monte Carlo with  std::reduce  +  par  execution policy. This uses OpenMP under the hood.